# Databricks: relational star schema to Delta

Generate dimension and fact tables for SQL joins and dashboard demos.

In [ ]:
# Databricks notebook source
%pip install great-generator

In [ ]:
from great_generator import generate_relational

data = generate_relational(
    tables={
        "dim_customer": {"schema": "customer_id int primary key, region string, customer_segment string", "rows": 10000},
        "dim_product": {"schema": "product_id int primary key, product_category string, brand string", "rows": 1000},
        "fact_sales": {"schema": "sales_id int primary key, customer_id int references dim_customer.customer_id, product_id int references dim_product.product_id, business_date date, sales_amount double, quantity int", "rows": 250000},
    },
    relationships=["fact_sales.customer_id -> dim_customer.customer_id", "fact_sales.product_id -> dim_product.product_id"],
    seed=42,
)

for name, pdf in data.items():
    spark.createDataFrame(pdf).write.format("delta").mode("overwrite").saveAsTable(f"default.{name}")